In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd

Q1_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(Q1_path)

In [ ]:
# Task 2: Write your code here:

print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Task 3: Write your code here:

df.info()

In [ ]:
# Task 4: Write your code here:

df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(df['Delivery_Time'].dropna())
plt.title('Delivery_Time Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_clean = df.drop('Order_ID', axis=1)
df_clean.head()

In [ ]:
missing_percentage = (df_clean.isnull().sum() / len(df_clean)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
# Task 2: Write your code here:
df_clean = df_clean.dropna(subset=['Weather','Traffic_Level','Time_of_Day','Courier_Experience_yrs']).copy()

df_clean['Delivery_Time'] = df_clean['Delivery_Time'].fillna(df['Delivery_Time'].mean()).copy()

print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder #import LabelEncoder

#print('data before encoding:\n', df_clean.head()) #show before encoding

label_encoder = LabelEncoder() # Instantiate LabelEncoder
#df_clean['Traffic_Level'] = label_encoder.fit_transform(df_clean["Traffic_Level"]) # Apply fit_transform to the column


cat_cols= df_clean.select_dtypes(include=["object"]).columns

one_hot = OneHotEncoder(sparse_output=False)
for col in cat_cols:
  df_clean[col] = label_encoder.fit_transform(df_clean[col])

print('\nData after encoding:\n', df_clean.head())

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler
features= ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs']

standard_scaler = StandardScaler() # Instantiate StandardScaler
df_clean[features] = standard_scaler.fit_transform(df_clean[features]) # Apply fit_transform

print('\nData after scaling:\n', df_clean) #show after scaling

In [ ]:
# Task 6: Write your code here:
#no need cause it's regression

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Delivery_Time",axis=1)
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Define K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

print(f"MAE: {mae:,.2f}")

In [ ]:
# Task 1: Write your code here:

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black', color='green')
plt.title('Prediction Hist')
plt.xlabel('Prediction')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: